In [3]:
import cv2
import numpy as np
import random
import math

def generate_grid_image(image_number):
    # 设置图像大小 - 增加分辨率
    width_5x5 = 600  # 增加宽度
    height_5x5 = 600  # 增加高度
    total_width = width_5x5 * 2 + 100  # 两个5x5网格加100像素的间隔

    # 创建一个空白图像 (白色背景)
    img = np.ones((height_5x5, total_width, 3), np.uint8) * 255  # 3通道，白色

    # 设置点的大小和颜色 - 根据新分辨率调整
    dot_size = 15  # 增大点的大小
    dot_color = (0, 0, 0)  # 黑色
    line_color = (0, 0, 0)  # 黑色线条
    line_thickness = 3  # 增加线条粗细

    # 计算5x5网格中点之间的间距
    x_spacing_5x5 = width_5x5 // 5
    y_spacing_5x5 = height_5x5 // 5

    # 创建左侧5x5网格的点坐标列表 (不绘制点，但需要坐标用于连线)
    points_left = []
    for i in range(5):  # i是行索引（对应y坐标）
        for j in range(5):  # j是列索引（对应x坐标）
            x = x_spacing_5x5 * j + x_spacing_5x5 // 2
            y = y_spacing_5x5 * i + y_spacing_5x5 // 2
            points_left.append((x, y))

    # 创建右侧5x5网格的点坐标列表
    points_right = []
    for i in range(5):
        for j in range(5):
            x = width_5x5 + 100 + x_spacing_5x5 * j + x_spacing_5x5 // 2  # 添加偏移
            y = y_spacing_5x5 * i + y_spacing_5x5 // 2
            points_right.append((x, y))

    # 在右侧5x5网格上绘制点
    for i, (x, y) in enumerate(points_right):
        cv2.circle(img, (x, y), dot_size // 2, dot_color, -1)

    # 随机选择起点
    start_index = random.randint(0, 24)
    start_point = points_left[start_index]
    print(f"图片 {image_number}: 起点是点 {start_index + 1}")

    # 生成随机线段
    num_segments = random.randint(4, 6)
    current_point = start_point
    current_index = start_index
    visited_indices = {start_index}  # 记录已经访问过的点的索引
    path = [current_point]  # 记录路径上的所有点
    path_indices = [current_index]  # 记录路径上的所有点索引

    def gcd(a, b):
        """计算最大公约数"""
        a, b = abs(a), abs(b)
        while b:
            a, b = b, a % b
        return a

    def are_coprime(a, b):
        """检查两个数是否互素（最大公约数为1）"""
        if a == 0 and b == 0:  # 特殊情况：0和0不互素
            return False
        return gcd(a, b) == 1

    def get_coprime_moves(row, col):
        """获取所有可能的互素移动"""
        moves = []
        # 考虑一个合理范围内的移动
        for i in range(-4, 5):  # 从-4到4
            for j in range(-4, 5):  # 从-4到4
                # 跳过原地不动的情况
                if i == 0 and j == 0:
                    continue

                # 检查i和j是否互素
                if are_coprime(i, j):
                    new_row, new_col = row + i, col + j
                    # 确保新位置在5x5网格内
                    if 0 <= new_row < 5 and 0 <= new_col < 5:
                        moves.append((new_row, new_col))
        return moves

    for _ in range(num_segments):
        # 获取当前点的行列索引
        current_row, current_col = divmod(current_index, 5)

        # 获取所有可能的互素移动
        possible_moves = get_coprime_moves(current_row, current_col)

        # 转换为点索引并过滤掉已访问过的点
        possible_next_indices = []
        for row, col in possible_moves:
            idx = row * 5 + col
            if idx not in visited_indices:
                possible_next_indices.append(idx)

        if not possible_next_indices:  # 如果没有可行的移动，结束循环
            print(f"图片 {image_number}: 没有可行的互素移动，结束连接。")
            break

        # 随机选择一个未访问的点
        next_index = random.choice(possible_next_indices)
        next_point = points_left[next_index]
        path.append(next_point)  # 添加到路径
        path_indices.append(next_index)  # 添加索引到路径

        # 更新当前点和索引
        current_point = next_point
        current_index = next_index
        visited_indices.add(next_index)  # 将新点添加到已访问集合

        # 计算移动的i和j值（用于调试）
        next_row, next_col = divmod(next_index, 5)
        i, j = next_row - current_row, next_col - current_col
        print(f"图片 {image_number}: 连接到点 {next_index + 1}，移动 ({i},{j})")

    # 在左侧绘制路径连线
    for i in range(len(path) - 1):
        cv2.line(img, path[i], path[i+1], line_color, line_thickness)

    # 在左侧5x5网格上标记起点
    cv2.circle(img, start_point, dot_size, (0, 0, 0), 2)  # 黑色圆圈
    cv2.circle(img, start_point, dot_size // 2, dot_color, -1)  # 在圆圈中心添加黑点

    # 在右侧5x5网格上标记与左侧5x5网格相同相对位置的起点
    start_point_right = points_right[start_index]

    # 在右侧5x5网格的起始点周围画一个圆圈，并在中心添加点
    cv2.circle(img, start_point_right, dot_size, (0, 0, 0), 2)  # 黑色圆圈
    cv2.circle(img, start_point_right, dot_size // 2, dot_color, -1)  # 在圆圈中心添加黑点

    # 保存图像 - 使用更高质量的保存设置
    filename = f"{image_number}.png"
    cv2.imwrite(filename, img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
    print(f"已保存图片: {filename}")

    # 返回终点索引（从1开始计数）
    end_index = path_indices[-1] + 1  # 添加1是因为我们要从1开始计数
    return img, end_index

# 主程序
def main():
    try:
        num_images = int(input("请输入要生成的图片数量: "))
        if num_images <= 0:
            print("请输入大于0的数字")
            return

        # 创建或覆盖ans.txt文件
        with open("ans.txt", "w") as f:
            for i in range(num_images):
                img, end_index = generate_grid_image(i)
                # 将终点写入ans.txt

                f.write(f"({end_index//5+1}, {end_index%5})\n")

            print(f"已成功生成 {num_images} 张图片，从0.png到{num_images-1}.png")
            print(f"所有终点已写入 ans.txt")
        
    except ValueError:
        print("请输入有效的数字")

if __name__ == "__main__":
    main()

图片 0: 起点是点 4
图片 0: 连接到点 20，移动 (3,1)
图片 0: 连接到点 1，移动 (-3,-4)
图片 0: 连接到点 2，移动 (0,1)
图片 0: 连接到点 7，移动 (1,0)
图片 0: 连接到点 21，移动 (3,-1)
已保存图片: 0.png
图片 1: 起点是点 6
图片 1: 连接到点 5，移动 (-1,4)
图片 1: 连接到点 24，移动 (4,-1)
图片 1: 连接到点 15，移动 (-2,1)
图片 1: 连接到点 10，移动 (-1,0)
图片 1: 连接到点 1，移动 (-1,-4)
已保存图片: 1.png
图片 2: 起点是点 20
图片 2: 连接到点 11，移动 (-1,-4)
图片 2: 连接到点 24，移动 (2,3)
图片 2: 连接到点 5，移动 (-4,1)
图片 2: 连接到点 7，移动 (1,-3)
图片 2: 连接到点 21，移动 (3,-1)
图片 2: 连接到点 17，移动 (-1,1)
已保存图片: 2.png
图片 3: 起点是点 15
图片 3: 连接到点 16，移动 (1,-4)
图片 3: 连接到点 21，移动 (1,0)
图片 3: 连接到点 2，移动 (-4,1)
图片 3: 连接到点 11，移动 (2,-1)
图片 3: 连接到点 7，移动 (-1,1)
图片 3: 连接到点 14，移动 (1,2)
已保存图片: 3.png
图片 4: 起点是点 17
图片 4: 连接到点 4，移动 (-3,2)
图片 4: 连接到点 5，移动 (0,1)
图片 4: 连接到点 7，移动 (1,-3)
图片 4: 连接到点 18，移动 (2,1)
图片 4: 连接到点 25，移动 (1,2)
图片 4: 连接到点 2，移动 (-4,-3)
已保存图片: 4.png
图片 5: 起点是点 16
图片 5: 连接到点 14，移动 (-1,3)
图片 5: 连接到点 6，移动 (-1,-3)
图片 5: 连接到点 3，移动 (-1,2)
图片 5: 连接到点 17，移动 (3,-1)
图片 5: 连接到点 18，移动 (0,1)
图片 5: 连接到点 9，移动 (-2,1)
已保存图片: 5.png
图片 6: 起点是点 15
图片 6: 连接到点 8，移动 (-1,-2)
图片 6: 连